In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders - FIXED BUG: test_loader was using train_dataset
batch_size = 4 # for saving time, my gpu end
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

import matplotlib.pyplot as plt
import numpy as np

# Define mean & std for denormalization
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Display 5 images
fig, axes = plt.subplots(1, 5, figsize=(15, 5))
imgs_indices = [270, 233, 110, 89, 15]

for i in range(5):
    img, label = train_dataset[imgs_indices[i]]
    img_np = img.numpy().transpose(1, 2, 0)
    img_np = std * img_np + mean
    img_np = np.clip(img_np, 0, 1)
    axes[i].imshow(img_np)
    axes[i].set_title(f'Label: {letters[label-1]}')
    axes[i].axis('off')

plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
from torchvision import models

model = models.efficientnet_v2_s(pretrained=True)
model.classifier[1] = nn.Linear(1280, 26)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(model)

In [ ]:
import torch.optim as optim
from tqdm import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for inputs, labels in tqdm(dataloader):
        labels = labels - 1

        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = (correct_predictions / total_samples) * 100
    return epoch_loss, epoch_accuracy

def validate_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader):
            labels = labels - 1

            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = (correct_predictions / total_samples) * 100
    return epoch_loss, epoch_accuracy

In [ ]:
# Training configuration
num_epochs = 1

# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

print(f"Starting training for {num_epochs} epochs...")
print(f"Device: {device}")
print("-" * 80)

# Training process
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy = validate_epoch(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Train Loss={train_loss:.4f}, Train Acc={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Acc={val_accuracy:.2f}%")

print("\nTraining completed!")

In [ ]:
# Plot training and validation losses and accuracies
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# losses
ax1.plot(range(1, num_epochs+1), train_losses, 'b-', label='Training Loss', linewidth=2)
ax1.plot(range(1, num_epochs+1), val_losses, 'r-', label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# accuracies
ax2.plot(range(1, num_epochs+1), train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
ax2.plot(range(1, num_epochs+1), val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Write your code here
def validate_with_tta(model, dataloader, criterion, device):
    """Validate with Test Time Augmentation"""
    model.eval()
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader):
            labels = labels - 1

            inputs, labels = inputs.to(device), labels.to(device)
            outputs_original = model(inputs)

            h_flipped = torch.flip(inputs, dims=[3])
            outputs_h_flipped = model(h_flipped)

            v_flipped = torch.flip(inputs, dims=[2])
            outputs_v_flipped = model(v_flipped)

            outputs_avg = (outputs_original + outputs_h_flipped + outputs_v_flipped) / 3.0

            loss = criterion(outputs_avg, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs_avg.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = (correct_predictions / total_samples) * 100
    return epoch_loss, epoch_accuracy


print("TTA")
tta_loss, tta_accuracy = validate_with_tta(model, test_loader, criterion, device)

print(f"\nResults:")
print(f"Standard Validationacc: {val_accuracies[-1]:.2f}%")
print(f"TTA Validation acc: {tta_accuracy:.2f}%")
print(f"TTA Improvemnt - acc: {tta_accuracy - val_accuracies[-1]:+.2f}%")